### <mark cololr='green'>1단계 : 데이터 준비</mark>



In [1]:
import numpy as np
from sklearn.datasets import make_moons

In [10]:
X, y = make_moons(n_samples=1000, shuffle=True, noise=0.2, random_state=42)

print(X.shape)
print(y.shape)

(1000, 2)
(1000,)


In [20]:
from sklearn.model_selection import train_test_split

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [22]:
from sklearn.preprocessing import StandardScaler

In [23]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)

### <mark>2단계 : 모델 설계</mark>

In [24]:
import torch

In [25]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [26]:
import torch.nn as nn

In [27]:
model = nn.Sequential(nn.Linear(2,16), nn.ReLU(), nn.Linear(16,2))

print(model)

Sequential(
  (0): Linear(in_features=2, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=2, bias=True)
)


### <mark>3단계 : 손실함수 + 옵티마이저</mark>

In [31]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

### <mark>4단계 : 학습 루프 </mark>

In [33]:
for epoch in range(1000):
  # 1) 이전 기울기 비우기
  optimizer.zero_grad()

  # 2) 순전파 - 데이터를 모델에 통과시켜 예측을 얻기
  outputs = model(X_train_tensor)

  # 3) 손실 계산 - 정답과 예측이 얼마나 틀렸는지
  loss = loss_fn(outputs, y_train_tensor)

  # 4) 역전파 - 각 가중치가 오차에 얼마나 책임있는지 기울기 확인
  loss.backward()

  # 5) 업데이트 - 기울기를 보고 가중치를 수정
  optimizer.step()

  # 100번 돌 때마다 손실이 얼마나 주는지 확인
  if epoch%100 == 0 :
    print(loss.item())

0.6284604668617249
0.21818268299102783
0.09738250076770782
0.07359103113412857
0.06720106303691864
0.06416234374046326
0.06241772323846817
0.06116461008787155
0.06012043356895447
0.05725662410259247


### <mark> 5단계 : 평가 </mark>

In [41]:
# 위에서 측정한 기울기를 가지고 모델에서 예측하기
with torch.no_grad():
  test_outputs = model(X_test_tensor)

In [42]:
# 행에서 예측값 중 큰 값의 위치 뽑기
predicted = torch.argmax(test_outputs, dim=1)

In [43]:
# predicted == y_test_tensor 가 맞으면 T, 틀리면 F
# float : T는 1.0, F는 0.0
# mean() : 전체의 평균
accuracy = (predicted == y_test_tensor).float().mean()

print(accuracy.item())

0.9760000109672546
